In [1]:
import duckdb
import boto3
import pandas as pd
from io import BytesIO

# Connect to LocalStack
s3 = boto3.client(
    "s3",
    aws_access_key_id="test",
    aws_secret_access_key="test",
    region_name="us-east-1",
    endpoint_url="http://localhost:4566"
)

bucket = "zazi-data-bucket"

# List files in stream-output
response = s3.list_objects_v2(Bucket=bucket, Prefix="stream-output/")
files = [obj["Key"] for obj in response.get("Contents", [])]
print("✅ Found Parquet files:", files)

# Read all parquet files into one dataframe
frames = []
for key in files:
    buffer = BytesIO()
    s3.download_fileobj(bucket, key, buffer)
    buffer.seek(0)
    df = pd.read_parquet(buffer)
    frames.append(df)

data = pd.concat(frames, ignore_index=True)
print("✅ Combined Data Preview:")
display(data.head())

# Query using DuckDB (Athena-style)
query = """
SELECT city, COUNT(*) AS total_tx, SUM(purchase_value_with_vat) AS total_spent
FROM data
GROUP BY city
ORDER BY total_spent DESC
"""
result = duckdb.query(query).to_df()

print("🏆 Aggregated City Spend:")
display(result)


✅ Found Parquet files: ['stream-output/zazi_stream_10.parquet', 'stream-output/zazi_stream_100.parquet', 'stream-output/zazi_stream_105.parquet', 'stream-output/zazi_stream_110.parquet', 'stream-output/zazi_stream_115.parquet', 'stream-output/zazi_stream_120.parquet', 'stream-output/zazi_stream_125.parquet', 'stream-output/zazi_stream_130.parquet', 'stream-output/zazi_stream_135.parquet', 'stream-output/zazi_stream_140.parquet', 'stream-output/zazi_stream_145.parquet', 'stream-output/zazi_stream_15.parquet', 'stream-output/zazi_stream_150.parquet', 'stream-output/zazi_stream_155.parquet', 'stream-output/zazi_stream_160.parquet', 'stream-output/zazi_stream_165.parquet', 'stream-output/zazi_stream_170.parquet', 'stream-output/zazi_stream_175.parquet', 'stream-output/zazi_stream_180.parquet', 'stream-output/zazi_stream_185.parquet', 'stream-output/zazi_stream_190.parquet', 'stream-output/zazi_stream_195.parquet', 'stream-output/zazi_stream_20.parquet', 'stream-output/zazi_stream_200.parqu

,customer_id,name,city,purchase_value,purchase_value_with_vat,segment
0,140,Amahle,Joburg,318,365.7,High
1,840,Amahle,Durban,450,517.5,High
2,807,Kwena,Pretoria,322,370.3,High
3,373,Nathi,Joburg,552,634.8,High
4,259,Zazi,Cape Town,508,584.2,High


🏆 Aggregated City Spend:


,city,total_tx,total_spent
0,Pretoria,10842,4202409.35
1,Durban,9291,3812192.50
2,Cape Town,10140,3795054.05
3,Joburg,9732,3766301.75
